# JAX Communication-Bit Curriculum

Start from the `25x25` one-bit forage policy produced by the 50-padded forage curriculum, then train staged communication alphabets. The notebook is a thin runner; checkpoint transfer, tqdm bookkeeping, rendering, and vault creation live in `ant_byte_env.notebook_workflows`.


In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
{"project_root": PROJECT_ROOT, **runtime_status}


In [ ]:
import importlib

import jax

from ant_byte_env import notebook_workflows as workflows
from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Curriculum Settings


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "communication_bits.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "communication_bits_25x25"
MEDIA_DIR = RUN_DIR / "media"
TRAINING_ARGS = dict(experiment.args)

SOURCE_CHECKPOINT = workflows.resolve_project_path(
    PROJECT_ROOT,
    experiment.metadata.get("source_checkpoint")
    or TRAINING_ARGS.get("load_model")
    or "runs/notebooks/forage_curriculum/checkpoints/jax_mappo_forage_stage1_25x25.pkl",
)
BIT_STAGES = [int(bits) for bits in experiment.metadata.get("bit_stages", [2, 3, 5, 8])]
GLOBAL_UPDATE_CAP = int(experiment.metadata.get("global_update_cap", 10000))
CONSOLIDATION_CONFIG = dict(experiment.metadata.get("consolidation", {}))
POLISH_CONFIG = dict(experiment.metadata.get("polish", {}))
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
NUM_ENVS = int(TRAINING_ARGS["num_envs"])
NUM_STEPS = int(TRAINING_ARGS["num_steps"])
UPDATE_TIMESTEPS = workflows.update_timesteps(num_envs=NUM_ENVS, num_steps=NUM_STEPS)
COMMON_ARGS = workflows.config_common_args(TRAINING_ARGS, exclude=workflows.COMMUNICATION_ARG_EXCLUDES)

workflows.validate_communication_stages(BIT_STAGES)
if not SOURCE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing source checkpoint: {SOURCE_CHECKPOINT}")

{
    "experiment_config": EXPERIMENT_CONFIG,
    "source_checkpoint": SOURCE_CHECKPOINT,
    "bit_stages": BIT_STAGES,
    "updates_per_stage": GLOBAL_UPDATE_CAP,
    "consolidation": CONSOLIDATION_CONFIG,
    "polish": POLISH_CONFIG,
}


## Train Bit Stages


In [ ]:
communication_result = workflows.run_communication_bit_curriculum(
    bit_stages=BIT_STAGES,
    source_checkpoint=SOURCE_CHECKPOINT,
    run_dir=RUN_DIR,
    common_args=COMMON_ARGS,
    experiment_name=experiment.args.get("exp_name", experiment.name),
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    global_update_cap=GLOBAL_UPDATE_CAP,
    train_main=jax_runner.main,
)
post_stage_result = workflows.run_communication_post_stage_sequence(
    stage_configs={"consolidation": CONSOLIDATION_CONFIG, "polish": POLISH_CONFIG},
    source_checkpoint=communication_result["final_checkpoint"],
    target_bits=BIT_STAGES[-1],
    run_dir=RUN_DIR,
    common_args=COMMON_ARGS,
    experiment_name=experiment.args.get("exp_name", experiment.name),
    update_timesteps_per_stage=UPDATE_TIMESTEPS,
    train_main=jax_runner.main,
)
FINAL_COMMUNICATION_CHECKPOINT = post_stage_result["final_checkpoint"]
CONSOLIDATED_CHECKPOINTS = post_stage_result["checkpoint_paths"]
post_stage_results = post_stage_result["stage_results"]

{"curriculum": communication_result, **post_stage_results}


## Optional Render and Vault


In [ ]:
rollout_result = workflows.render_communication_rollouts(
    experiment_config=EXPERIMENT_CONFIG,
    source_checkpoint=SOURCE_CHECKPOINT,
    run_dir=RUN_DIR,
    media_dir=MEDIA_DIR,
    bit_stages=BIT_STAGES,
    global_update_cap=GLOBAL_UPDATE_CAP,
    extra_checkpoint_paths=CONSOLIDATED_CHECKPOINTS,
    tile_size=ROLLOUT_TILE_SIZE,
)
rollout_result
